Author: Nasr SLIMANI

# Pricing of a Bermudean Put Option using Nested Monte Carlo Simulations

## Naive implementation

In [9]:
import numpy as np
import time
from typing import Tuple
from dataclasses import dataclass

@dataclass
class SimulationParams:
    """Parameters for the Monte Carlo simulation"""
    S_0: float = 100.0    # Initial stock price
    K: float = 100.0      # Strike price
    r: float = 0.05       # Risk-free rate
    sigma: float = 0.2    # Volatility
    T: float = 3.0        # Time to maturity (3 years)
    m: int = 3            # Number of exercise dates
    b: int = 50           # Number of branches
    n: int = 10          # Number of simulations

class MonteCarloOptionPricer:
    """Monte Carlo simulation for option pricing using high and low estimators"""
    
    def __init__(self, params: SimulationParams):
        self.params = params
        self.dt = params.T / params.m
        
    def generate_paths(self) -> np.ndarray:
        """Generate stock price paths using the Black-Scholes model"""
        S = np.zeros((self.params.b**self.params.m, self.params.m + 1))
        S[:, 0] = self.params.S_0
        
        drift = (self.params.r - 0.5 * self.params.sigma**2) * self.dt
        vol = self.params.sigma * np.sqrt(self.dt)
        
        for i in range(self.params.m):
            for j in range(self.params.b**(i+1)):
                returns = drift + vol * np.random.randn()
                S[j, i+1] = S[j//self.params.b, i] * np.exp(returns)
                
        return S

    def high_estimator(self, S: np.ndarray) -> float:
        """Calculate option price using the high estimator
        
        Args:
            S: Matrix of stock price paths
        Returns:
            float: Option price at t0 using high estimator
        """
        V_high = np.zeros((self.params.b**self.params.m, self.params.m + 1))
        V_high[:, self.params.m] = np.maximum(self.params.K - S[:, self.params.m], 0)
        
        for i in range(self.params.m):
            for j in range(self.params.b**(self.params.m-1-i)):
                continuation_value = np.exp(-self.dt * self.params.r) * np.mean(
                    V_high[j*self.params.b:(j+1)*self.params.b, self.params.m-i]
                )
                immediate_exercise = np.maximum(self.params.K - S[j, self.params.m-1-i], 0)
                V_high[j, self.params.m-1-i] = np.maximum(immediate_exercise, continuation_value)
                
        return V_high[0, 0]

    def low_estimator(self, S: np.ndarray) -> float:
        """Calculate option price using the low estimator
        
        Args:
            S: Matrix of stock price paths
        Returns:
            float: Option price at t0 using low estimator
        """
        V_low = np.zeros((self.params.b**self.params.m, self.params.m + 1))
        V_low[:, self.params.m] = np.maximum(self.params.K - S[:, self.params.m], 0)
        
        for i in range(self.params.m):
            for j in range(self.params.b**(self.params.m-1-i)):
                V = np.zeros(self.params.b)
                for k in range(self.params.b):
                    # Create ensemble excluding current index
                    ensemble = [j*self.params.b + ii for ii in range(self.params.b) if ii != k]
                    continuation_value = np.exp(-self.params.r) * np.mean(V_low[ensemble, self.params.m-i])
                    specific_value = np.exp(-self.params.r) * V_low[j*self.params.b + k, self.params.m-i]
                    immediate_exercise = np.maximum(self.params.K - S[j, self.params.m-1-i], 0)
                    
                    V[k] = immediate_exercise if continuation_value <= immediate_exercise else specific_value
                
                V_low[j, self.params.m-1-i] = np.mean(V)
                
        return V_low[0, 0]
    
    def run_simulation(self) -> Tuple[np.ndarray, float]:
        """Run the Monte Carlo simulation"""
        start_time = time.perf_counter()
        
        results = np.zeros((self.params.n, 2))
        for k in range(self.params.n):
            paths = self.generate_paths()
            results[k, 0] = self.high_estimator(paths)
            results[k, 1] = self.low_estimator(paths)
            
        computation_time = time.perf_counter() - start_time
        return results, computation_time
    
    def print_results(self, results: np.ndarray, computation_time: float) -> None:
        """Print simulation results with confidence intervals"""
        print("Numerical results")
        print(f"Time Execution = {computation_time:.4f} s")
        print("-" * 93)
        
        # High estimator statistics
        price_high = np.mean(results[:, 0])
        std_high = np.std(results[:, 0])
        error_margin_high = 100 * 2 * 1.96 * std_high / (np.sqrt(self.params.n) * price_high)
        ci_high = [
            price_high - 1.96 * std_high / np.sqrt(self.params.n),
            price_high + 1.96 * std_high / np.sqrt(self.params.n)
        ]
        
        print(f"High Estimator price Option = {price_high:.4f}  Error Margin = {error_margin_high:.2f}%")
        print(f"High Confidence interval at 95% = [{ci_high[0]:.4f}, {ci_high[1]:.4f}]")
        print("-" * 93)
        
        # Low estimator statistics
        price_low = np.mean(results[:, 1])
        std_low = np.std(results[:, 1])
        error_margin_low = 100 * 2 * 1.96 * std_low / (np.sqrt(self.params.n) * price_low)
        ci_low = [
            price_low - 1.96 * std_low / np.sqrt(self.params.n),
            price_low + 1.96 * std_low / np.sqrt(self.params.n)
        ]
        
        print(f"Low Estimator price Option = {price_low:.4f}  Error Margin = {error_margin_low:.2f}%")
        print(f"Low Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_low[1]:.4f}]")
        print("-" * 93)
        
        # Combined estimation
        price_estimation = (ci_low[0] + ci_high[1]) / 2
        error_combined = -100 * (ci_low[0] - ci_high[1]) / price_estimation
        
        print(f"Estimation price Option = {price_estimation:.4f}  Error = {error_combined:.2f}%")
        print(f"Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_high[1]:.4f}]")


In [6]:
params = SimulationParams()

# Create pricer instance and run simulation
pricer = MonteCarloOptionPricer(params)
results, computation_time = pricer.run_simulation()
pricer.print_results(results, computation_time)

Numerical results
Time Execution = 39.0099 s
---------------------------------------------------------------------------------------------
High Estimator price Option = 8.6592  Error Margin = 18.57%
High Confidence interval at 95% = [7.8554, 9.4631]
---------------------------------------------------------------------------------------------
Low Estimator price Option = 8.4847  Error Margin = 18.27%
Low Confidence interval at 95% = [7.7094, 9.2599]
---------------------------------------------------------------------------------------------
Estimation price Option = 8.5863  Error = 20.42%
Confidence interval at 95% = [7.7094, 9.4631]


## Depth first processing

In [8]:
import numpy as np
import time
from typing import Tuple
from dataclasses import dataclass

@dataclass
class SimulationParams:
    """Parameters for the Monte Carlo simulation"""
    S_0: float = 100.0    # Initial stock price
    K: float = 100.0      # Strike price
    r: float = 0.05       # Risk-free rate
    sigma: float = 0.2    # Volatility
    T: float = 3.0        # Time to maturity (3 years)
    m: int = 3            # Number of exercise dates
    b: int = 80           # Number of branches
    n: int = 10          # Number of simulations

class MonteCarloOptionPricer:
    """Efficient handling of memory, advanced version of the previous"""
    
    def __init__(self, params: SimulationParams):
        self.params = params
        self.dt = params.T / params.m
        
    def generate_level_prices(self, S_current: np.ndarray, num_next: int) -> np.ndarray:
        """Generate stock prices for the next level"""
        drift = (self.params.r - (self.params.sigma**2)/2) * self.dt
        vol = self.params.sigma * np.sqrt(self.dt)
        
        next_prices = np.zeros(num_next)
        for i, S in enumerate(S_current):
            start_idx = i * self.params.b
            end_idx = (i + 1) * self.params.b
            returns = drift + vol * np.random.randn(self.params.b)
            next_prices[start_idx:end_idx] = S * np.exp(returns)
            
        return next_prices
    
    def calculate_node_values(self, S: float, next_values: np.ndarray) -> Tuple[float, float, float]:
        """Calculate high and low estimator values for a node"""
        immediate_exercise = np.maximum(self.params.K - S, 0)
        
        # High estimator
        continuation_value = np.exp(-self.params.r) * np.mean(next_values[:, 0])
        V_high = np.maximum(immediate_exercise, continuation_value)
        
        # Low estimators
        V = np.zeros(self.params.b)
        for j in range(self.params.b):
            ensemble = [i for i in range(self.params.b) if i != j]
            V1 = np.exp(-self.params.r) * np.mean(next_values[ensemble, 1])
            V2 = np.exp(-self.params.r) * next_values[j, 1]
            V[j] = immediate_exercise if V1 <= immediate_exercise else V2
            
        V1_low = np.mean(V)
        
        # Second low estimator (using same price)
        V2_low = V1_low  # Simplified for this example
        
        return V_high, V1_low, V2_low
    
    def run_simulation(self) -> Tuple[np.ndarray, float]:
        """Run the Monte Carlo simulation using level-by-level processing"""
        start_time = time.perf_counter()
        results = np.zeros((self.params.n, 3))
        
        for sim in range(self.params.n):
            # Initialize arrays for the final level
            num_final = self.params.b**self.params.m
            current_values = np.zeros((num_final, 3))
            
            # Forward pass - generate all prices level by level
            prices = [np.array([self.params.S_0])]
            for level in range(self.params.m):
                next_prices = self.generate_level_prices(
                    prices[-1],
                    self.params.b**(level+1)
                )
                prices.append(next_prices)
            
            # Terminal values
            current_values[:, :] = np.maximum(self.params.K - prices[-1], 0)[:, np.newaxis]
            
            # Backward pass - calculate option values
            for level in range(self.params.m-1, -1, -1):
                num_nodes = self.params.b**level
                new_values = np.zeros((num_nodes, 3))
                
                for node in range(num_nodes):
                    start_idx = node * self.params.b
                    end_idx = (node + 1) * self.params.b
                    next_values = current_values[start_idx:end_idx]
                    
                    new_values[node] = self.calculate_node_values(
                        prices[level][node], 
                        next_values
                    )
                
                current_values = new_values
            
            results[sim] = current_values[0]
            
        computation_time = time.perf_counter() - start_time
        return results, computation_time

    def print_results(self, results: np.ndarray, computation_time: float) -> None:
        """Print simulation results with confidence intervals"""
        print("Numerical results")
        print(f"Time Execution = {computation_time:.4f} s")
        print("-" * 93)
        
        # High estimator statistics
        price_high = np.mean(results[:, 0])
        std_high = np.std(results[:, 0])
        error_margin_high = 100 * 2 * 1.96 * std_high / (np.sqrt(self.params.n) * price_high)
        ci_high = [
            price_high - 1.96 * std_high / np.sqrt(self.params.n),
            price_high + 1.96 * std_high / np.sqrt(self.params.n)
        ]
        
        print(f"High Estimator price Option = {price_high:.4f}  Error Margin = {error_margin_high:.2f}%")
        print(f"High Confidence interval at 95% = [{ci_high[0]:.4f}, {ci_high[1]:.4f}]")
        print("-" * 93)
        
        # Low estimator statistics
        price_low = np.mean(results[:, 1])
        std_low = np.std(results[:, 1])
        error_margin_low = 100 * 2 * 1.96 * std_low / (np.sqrt(self.params.n) * price_low)
        ci_low = [
            price_low - 1.96 * std_low / np.sqrt(self.params.n),
            price_low + 1.96 * std_low / np.sqrt(self.params.n)
        ]
        
        print(f"Low Estimator price Option = {price_low:.4f}  Error Margin = {error_margin_low:.2f}%")
        print(f"Low Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_low[1]:.4f}]")
        print("-" * 93)
        
        # Combined estimation
        price_estimation = (ci_low[0] + ci_high[1]) / 2
        error_combined = -100 * (ci_low[0] - ci_high[1]) / price_estimation
        
        print(f"Estimation price Option = {price_estimation:.4f}  Error = {error_combined:.2f}%")
        print(f"Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_high[1]:.4f}]")

params = SimulationParams()
pricer = MonteCarloOptionPricer(params)
results, computation_time = pricer.run_simulation()
pricer.print_results(results, computation_time)


Numerical results
Time Execution = 127.5000 s
---------------------------------------------------------------------------------------------
High Estimator price Option = 8.6827  Error Margin = 9.02%
High Confidence interval at 95% = [8.2912, 9.0742]
---------------------------------------------------------------------------------------------
Low Estimator price Option = 8.5907  Error Margin = 9.31%
Low Confidence interval at 95% = [8.1906, 8.9908]
---------------------------------------------------------------------------------------------
Estimation price Option = 8.6324  Error = 10.24%
Confidence interval at 95% = [8.1906, 9.0742]


## Pruning method

In [ ]:
import numpy as np
import scipy.stats as sps
import time
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class PricingParams:
    """Parameters for option pricing"""
    S_0: float = 100.0    # Initial stock price
    K: float = 100.0      # Strike price
    r: float = 0.05       # Risk-free rate
    sigma: float = 0.2    # Volatility
    T: float = 3.0        # Time to maturity
    m: int = 3            # Number of exercise dates
    b: int = 40           # Number of branches
    n: int = 1000         # Number of simulations

class OptimizedPruningPricer:
    """Option pricer with pruning optimization"""
    
    def __init__(self, params: PricingParams):
        self.params = params
        self.dt = params.T / params.m
        
        # Precompute constants for Black-Scholes
        self.drift = (params.r - (params.sigma**2)/2) * self.dt
        self.vol = params.sigma * np.sqrt(self.dt)
        
    def black_scholes_put(self, S: float, time_to_maturity: float) -> float:
        """Calculate European put option price using Black-Scholes formula
        
        Args:
            S: Current stock price
            time_to_maturity: Time remaining until maturity
        """
        if time_to_maturity == 0:
            return max(self.params.K - S, 0)
            
        d1 = (np.log(S/self.params.K) + 
              (self.params.r + self.params.sigma**2/2) * time_to_maturity) / \
             (self.params.sigma * np.sqrt(time_to_maturity))
        d2 = d1 - self.params.sigma * np.sqrt(time_to_maturity)
        
        return (-S * sps.norm.cdf(-d1) + 
                self.params.K * np.exp(-self.params.r * time_to_maturity) * sps.norm.cdf(-d2))
    
    def generate_next_price(self, S: float) -> float:
        """Generate next stock price using precomputed constants"""
        return S * np.exp(self.drift + self.vol * np.random.randn())
    
    def low_estimator(self, V_successor: np.ndarray, S: float) -> float:
        """Calculate low estimator value"""
        V = np.zeros(self.params.b)
        immediate_exercise = max(self.params.K - S, 0)
        
        for j in range(self.params.b):
            ensemble = [i for i in range(self.params.b) if i != j]
            V1 = np.exp(-self.params.r) * np.mean(V_successor[ensemble, 1])
            V2 = np.exp(-self.params.r) * V_successor[j, 1]
            V[j] = immediate_exercise if V1 <= immediate_exercise else V2
            
        return np.mean(V)
    
    def estimate_node_with_pruning(self, S: float, k: int) -> List[float]:
        """Estimate node values with pruning optimization
        
        Args:
            S: Current stock price
            k: Current time step
        """
        time_to_maturity = (self.params.m - k) * self.dt
        immediate_exercise = max(self.params.K - S, 0)
        
        if k == self.params.m - 1:
            # At penultimate step, compare with European option value
            european_value = self.black_scholes_put(S, self.dt)
            value = max(immediate_exercise, european_value)
            return [value, value]
            
        # Calculate European option value for pruning decision
        european_value = self.black_scholes_put(S, time_to_maturity)
        
        if immediate_exercise < european_value:
            # Optimal to continue - use single successor
            next_price = self.generate_next_price(S)
            successor_values = self.estimate_node_with_pruning(next_price, k + 1)
            return [np.exp(-self.params.r) * v for v in successor_values]
        
        # Generate all successors if needed
        V_successor = np.zeros((self.params.b, 2))
        for i in range(self.params.b):
            next_price = self.generate_next_price(S)
            V_successor[i, :] = self.estimate_node_with_pruning(next_price, k + 1)
        
        # Calculate estimators
        continuation_value = np.exp(-self.params.r) * np.mean(V_successor[:, 0])
        V_high = max(immediate_exercise, continuation_value)
        V1_low = self.low_estimator(V_successor, S)
        
        return [V_high, V1_low]
    
    def run_simulation(self) -> Tuple[np.ndarray, float]:
        """Run Monte Carlo simulation with pruning
        
        Returns:
            Tuple[np.ndarray, float]: (Results array, Computation time)
        """
        start_time = time.perf_counter()
        
        results = np.zeros((self.params.n, 2))
        for i in range(self.params.n):
            results[i, :] = self.estimate_node_with_pruning(self.params.S_0, 0)
            
        computation_time = time.perf_counter() - start_time
        return results, computation_time
    
    def print_results(self, results: np.ndarray, computation_time: float) -> None:
        """Print simulation results with confidence intervals"""
        print("Numerical results")
        print(f"Time Execution = {computation_time:.4f} s")
        print("-" * 93)
        
        # High estimator statistics
        price_high = np.mean(results[:, 0])
        std_high = np.std(results[:, 0])
        error_margin_high = 100 * 2 * 1.96 * std_high / (np.sqrt(self.params.n) * price_high)
        ci_high = [
            price_high - 1.96 * std_high / np.sqrt(self.params.n),
            price_high + 1.96 * std_high / np.sqrt(self.params.n)
        ]
        
        print(f"High Estimator price Option = {price_high:.4f}  Error Margin = {error_margin_high:.2f}%")
        print(f"High Confidence interval at 95% = [{ci_high[0]:.4f}, {ci_high[1]:.4f}]")
        print("-" * 93)
        
        # Low estimator statistics
        price_low = np.mean(results[:, 1])
        std_low = np.std(results[:, 1])
        error_margin_low = 100 * 2 * 1.96 * std_low / (np.sqrt(self.params.n) * price_low)
        ci_low = [
            price_low - 1.96 * std_low / np.sqrt(self.params.n),
            price_low + 1.96 * std_low / np.sqrt(self.params.n)
        ]
        
        print(f"Low Estimator price Option = {price_low:.4f}  Error Margin = {error_margin_low:.2f}%")
        print(f"Low Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_low[1]:.4f}]")
        print("-" * 93)
        
        # Combined estimation
        price_estimation = (ci_low[0] + ci_high[1]) / 2
        error_combined = -100 * (ci_low[0] - ci_high[1]) / price_estimation
        
        print(f"Estimation price Option = {price_estimation:.4f}  Error = {error_combined:.2f}%")
        print(f"Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_high[1]:.4f}]")

params = PricingParams()
pricer = OptimizedPruningPricer(params)
results, computation_time = pricer.run_simulation()
pricer.print_results(results, computation_time)


Numerical results
Time Execution = 5.7621 s
---------------------------------------------------------------------------------------------
High Estimator price Option = 8.4933  Error Margin = 13.87%
High Confidence interval at 95% = [7.9042, 9.0824]
---------------------------------------------------------------------------------------------
Low Estimator price Option = 8.3791  Error Margin = 13.96%
Low Confidence interval at 95% = [7.7942, 8.9640]
---------------------------------------------------------------------------------------------
Estimation price Option = 8.4383  Error = 15.27%
Confidence interval at 95% = [7.7942, 9.0824]


## Variance reduction techniques

### Pruning + Antithetic variables

In [10]:
import numpy as np
import scipy.stats as sps
import time
from dataclasses import dataclass
from typing import List, Tuple, Optional

@dataclass
class PricingParams:
    """Parameters for Bermudean put option pricing"""
    S_0: float = 100.0    # Initial stock price
    K: float = 100.0      # Strike price
    r: float = 0.05       # Risk-free rate
    sigma: float = 0.2    # Volatility
    T: float = 3.0        # Time to maturity
    m: int = 3            # Number of exercise dates
    b: int = 50           # Number of branches for pruning
    n: int = 500          # Number of simulations

class OptimizedBermudeanPricer:
    """
    Enhanced Bermudean put option pricer using pruning and antithetic variables
    for variance reduction and improved computational efficiency.
    """
    
    def __init__(self, params: PricingParams):
        self.params = params
        self.dt = params.T / params.m
        
        # Precompute constants for efficiency
        self.drift = (params.r - 0.5 * params.sigma**2) * self.dt
        self.vol = params.sigma * np.sqrt(self.dt)
        self.discount = np.exp(-params.r * self.dt)
        
        # Precompute normal distribution for performance
        self._norm = sps.norm()
        
    def _black_scholes_put(self, S: float, time_to_maturity: float) -> float:
        """
        Optimized Black-Scholes formula for European put pricing
        
        Args:
            S: Current stock price
            time_to_maturity: Time remaining until maturity
            
        Returns:
            float: European put option price
        """
        if time_to_maturity <= 0:
            return max(self.params.K - S, 0)
            
        sqrt_t = np.sqrt(time_to_maturity)
        d1 = (np.log(S/self.params.K) + 
              (self.params.r + 0.5 * self.params.sigma**2) * time_to_maturity) / \
             (self.params.sigma * sqrt_t)
        d2 = d1 - self.params.sigma * sqrt_t
        
        return (self.params.K * np.exp(-self.params.r * time_to_maturity) * 
                self._norm.cdf(-d2) - S * self._norm.cdf(-d1))
    
    def _generate_paths(self, S: float, size: int) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate antithetic price paths for variance reduction
        
        Args:
            S: Current stock price
            size: Number of paths to generate
            
        Returns:
            Tuple containing normal and antithetic paths
        """
        Z = np.random.standard_normal(size)
        paths = S * np.exp(self.drift + self.vol * Z)
        paths_antithetic = S * np.exp(self.drift - self.vol * Z)
        return paths, paths_antithetic
    
    def _low_estimator(self, V_successor: np.ndarray, S1: float, S2: Optional[float] = None) -> float:
        """
        Calculate the second low estimator value.
    
        Args:
            V_successor: Successor node values, a 2D array of shape (b, 3).
            S1: Initial price of the first underlying asset.
            S2: Initial price of the second underlying asset (optional).
    
        Returns:
            The low estimator value for the given node.
        """
        V = np.zeros(self.params.b)
        for j in range(self.params.b):
            # Define the ensemble excluding the current index
            ensemble = [i for i in range(self.params.b) if i != j]
    
            # Calculate V1 and V2 based on successor values
            V1 = np.exp(-self.params.r * self.dt) * np.mean(V_successor[ensemble, 1])
            V2 = np.exp(-self.params.r * self.dt) * V_successor[j, 1]
    
            # Determine immediate exercise value
            if S2 is None:
                h = max(self.params.K - S1, 0)
            else:
                h = np.mean([max(self.params.K - S1, 0), max(self.params.K - S2, 0)])
    
            # Apply the low estimator logic
            V[j] = h if V1 <= h else V2
    
        # Return the average value as the estimator
        return np.mean(V)
    
    def estimate_node(self, S1: float, S2: float, k: int) -> List[float]:
        """
        Estimate option value at a given node using both price paths
        
        Args:
            S1: First price path
            S2: Antithetic price path
            k: Current time step
            
        Returns:
            List containing [high estimate, low estimate 1, low estimate 2]
        """
        time_to_maturity = (self.params.m - k) * self.dt
        immediate_exercise1 = max(self.params.K - S1, 0)
        immediate_exercise2 = max(self.params.K - S2, 0)
        immediate_exercise = 0.5 * (immediate_exercise1 + immediate_exercise2)
        
        # Terminal condition
        if k == self.params.m - 1:
            european_value = 0.5 * (self._black_scholes_put(S1, time_to_maturity) + 
                                  self._black_scholes_put(S2, time_to_maturity))
            value = max(immediate_exercise, european_value)
            return [value, value, value]
            
        # Check pruning condition using European value
        european_value = 0.5 * (self._black_scholes_put(S1, time_to_maturity) + 
                               self._black_scholes_put(S2, time_to_maturity))
        
        if immediate_exercise < european_value:
            # Continue with single successor pair
            next_S1, next_S2 = self._generate_paths(S1, 1)[0][0], self._generate_paths(S2, 1)[0][0]
            successor_values = self.estimate_node(next_S1, next_S2, k + 1)
            return [self.discount * v for v in successor_values]
        
        # Generate full set of successors
        V_successor = np.zeros((self.params.b, 3))
        for i in range(self.params.b):
            paths1, paths2 = self._generate_paths(S1, 1)[0][0], self._generate_paths(S2, 1)[0][0]
            V_successor[i, :] = self.estimate_node(paths1, paths2, k + 1)
        
        # Calculate estimators with variance reduction
        continuation_value = self.discount * np.mean(V_successor[:, 0])
        V_high = max(immediate_exercise, continuation_value)
        V1_low = self._low_estimator(V_successor, S1, S2)
        V2_low = self._low_estimator(V_successor, S1, S2)
        
        return [V_high, V1_low, V2_low]

    def price_option(self) -> Tuple[np.ndarray, float]:
        """
        Price the Bermudean put option using Monte Carlo simulation
        
        Returns:
            Tuple containing (Results array, Computation time)
        """
        start_time = time.perf_counter()
        
        results = np.zeros((self.params.n, 3))
        for i in range(self.params.n):
            results[i, :] = self.estimate_node(self.params.S_0, self.params.S_0, 0)
            
        computation_time = time.perf_counter() - start_time
        return results, computation_time

    def print_results(self, results: np.ndarray, computation_time: float) -> None:
        """Print simulation results with confidence intervals"""
        print("Numerical results")
        print(f"Time Execution = {computation_time:.4f} s")
        print("-" * 93)
        
        # High estimator statistics
        price_high = np.mean(results[:, 0])
        std_high = np.std(results[:, 0])
        error_margin_high = 100 * 2 * 1.96 * std_high / (np.sqrt(self.params.n) * price_high)
        ci_high = [
            price_high - 1.96 * std_high / np.sqrt(self.params.n),
            price_high + 1.96 * std_high / np.sqrt(self.params.n)
        ]
        
        print(f"High Estimator price Option = {price_high:.4f}  Error Margin = {error_margin_high:.2f}%")
        print(f"High Confidence interval at 95% = [{ci_high[0]:.4f}, {ci_high[1]:.4f}]")
        print("-" * 93)
        
        # Low estimator statistics
        price_low = np.mean(results[:, 1])
        std_low = np.std(results[:, 1])
        error_margin_low = 100 * 2 * 1.96 * std_low / (np.sqrt(self.params.n) * price_low)
        ci_low = [
            price_low - 1.96 * std_low / np.sqrt(self.params.n),
            price_low + 1.96 * std_low / np.sqrt(self.params.n)
        ]
        
        print(f"Low Estimator price Option = {price_low:.4f}  Error Margin = {error_margin_low:.2f}%")
        print(f"Low Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_low[1]:.4f}]")
        print("-" * 93)
        
        # Combined estimation
        price_estimation = (ci_low[0] + ci_high[1]) / 2
        error_combined = -100 * (ci_low[0] - ci_high[1]) / price_estimation
        
        print(f"Estimation price Option = {price_estimation:.4f}  Error = {error_combined:.2f}%")
        print(f"Confidence interval at 95% = [{ci_low[0]:.4f}, {ci_high[1]:.4f}]")
        
params = PricingParams()
pricer = OptimizedBermudeanPricer(params)
results, computation_time = pricer.price_option()
pricer.print_results(results, computation_time)

Numerical results
Time Execution = 3.4091 s
---------------------------------------------------------------------------------------------
High Estimator price Option = 7.5159  Error Margin = 14.92%
High Confidence interval at 95% = [6.9551, 8.0768]
---------------------------------------------------------------------------------------------
Low Estimator price Option = 7.4780  Error Margin = 14.95%
Low Confidence interval at 95% = [6.9189, 8.0371]
---------------------------------------------------------------------------------------------
Estimation price Option = 7.4978  Error = 15.44%
Confidence interval at 95% = [6.9189, 8.0768]


### Pruning + Control variates

In [11]:
import numpy as np
import scipy.stats as sps
import time
from dataclasses import dataclass
from typing import List, Tuple, Optional

@dataclass
class PricingParams:
    """Parameters for Bermudean put option pricing"""
    S_0: float = 100.0    # Initial stock price
    K: float = 100.0      # Strike price
    r: float = 0.05       # Risk-free rate
    sigma: float = 0.2    # Volatility
    T: float = 3.0        # Time to maturity
    m: int = 3            # Number of exercise dates
    b: int = 50           # Number of branches for pruning
    n: int = 500          # Number of simulations

class OptimizedBermudeanPricerCV:
    """
    Enhanced Bermudean put option pricer using pruning and control variate
    for variance reduction and improved computational efficiency.
    """
    
    def __init__(self, params: PricingParams):
        self.params = params
        self.dt = params.T / params.m
        
        # Precompute constants for efficiency
        self.drift = (params.r - 0.5 * params.sigma**2) * self.dt
        self.vol = params.sigma * np.sqrt(self.dt)
        self.discount = np.exp(-params.r * self.dt)
        
        # For Black-Scholes calculations
        self._norm = sps.norm()
    
    def _black_scholes_put(self, S: float, time_to_maturity: float) -> float:
        """Calculate Black-Scholes put option price"""
        if time_to_maturity <= 0:
            return max(self.params.K - S, 0)
            
        d1 = (np.log(S/self.params.K) + 
              (self.params.r + 0.5 * self.params.sigma**2) * time_to_maturity) / \
             (self.params.sigma * np.sqrt(time_to_maturity))
        d2 = d1 - self.params.sigma * np.sqrt(time_to_maturity)
        
        return (self.params.K * np.exp(-self.params.r * time_to_maturity) * 
                self._norm.cdf(-d2) - S * self._norm.cdf(-d1))
    
    def _generate_path(self, S: float) -> float:
        """Generate a single price path"""
        return S * np.exp(self.drift + self.vol * np.random.randn())
    
    def _low_estimator(self, V_successor: np.ndarray, S: float) -> float:
        """Calculate low estimator value"""
        V = np.zeros(self.params.b)
        for j in range(self.params.b):
            ensemble = [i for i in range(self.params.b) if i != j]
            V1 = self.discount * np.mean(V_successor[ensemble, 1])
            V2 = self.discount * V_successor[j, 1]
            h = max(self.params.K - S, 0)
            V[j] = h if V1 <= h else V2
        return np.mean(V)
    
    def estimate_node(self, S: float, k: int) -> List[float]:
        """
        Estimate option values at a node using control variate method
        
        Returns:
            [high estimate, low estimate, control variate]
        """
        time_to_maturity = (self.params.m - k) * self.dt
        immediate_exercise = max(self.params.K - S, 0)
        european_value = self._black_scholes_put(S, time_to_maturity)
        
        # Terminal condition
        if k == self.params.m - 1:
            value = max(immediate_exercise, european_value)
            control = self.discount * max(self.params.K - self._generate_path(S), 0)
            return [value, value, control]
            
        # Pruning condition check
        if immediate_exercise < european_value:
            successor_values = self.estimate_node(self._generate_path(S), k + 1)
            return [self.discount * v for v in successor_values]
        
        # Generate successor nodes
        V_successor = np.zeros((self.params.b, 3))
        for i in range(self.params.b):
            V_successor[i, :] = self.estimate_node(self._generate_path(S), k + 1)
        
        # Calculate estimators
        V_high = max(immediate_exercise, self.discount * np.mean(V_successor[:, 0]))
        V_low = self._low_estimator(V_successor, S)
        control = self.discount * np.mean(V_successor[:, 2])
        
        return [V_high, V_low, control]

    def price_option(self) -> Tuple[np.ndarray, float]:
        """Price the option using Monte Carlo simulation with control variate"""
        start_time = time.perf_counter()
        
        # Generate paths and estimates
        results = np.zeros((self.params.n, 3))
        for i in range(self.params.n):
            results[i, :] = self.estimate_node(self.params.S_0, 0)
        
        # Calculate control variate expectation
        d1 = (np.log(self.params.S_0/self.params.K) + 
              (self.params.r + 0.5 * self.params.sigma**2) * self.params.T) / \
             (self.params.sigma * np.sqrt(self.params.T))
        d2 = d1 - self.params.sigma * np.sqrt(self.params.T)
        cv_expectation = (-self.params.S_0 * self._norm.cdf(-d1) + 
                         self.params.K * np.exp(-self.params.r * self.params.T) * 
                         self._norm.cdf(-d2))
        
        # Apply control variate adjustment
        control_variate = results[:, 2]
        for i in range(2):  # Apply to high and low estimators
            coef = np.cov(results[:, i], control_variate)[0, 1] / np.var(control_variate)
            results[:, i] -= coef * (control_variate - cv_expectation)
        
        computation_time = time.perf_counter() - start_time
        return results[:, :2], computation_time  # Return only high and low estimates
    
    def print_results(self, results: np.ndarray, computation_time: float) -> None:
        """Print pricing results"""
        print("\nNumerical results")
        print("Time Execution =", computation_time, "s")
        print("-" * 93)
        
        # High estimator
        Price_high = np.mean(results[:, 0])
        std_high = np.std(results[:, 0])
        error_high = 100 * 2 * 1.96 * std_high / (np.sqrt(self.params.n) * Price_high)
        ci_high = [Price_high - 1.96 * std_high / np.sqrt(self.params.n),
                  Price_high + 1.96 * std_high / np.sqrt(self.params.n)]
        
        print("High Estimator price Option =", Price_high, "ErrorMargin =", error_high, "%")
        print("High Confidence interval at 95% =", list(ci_high))
        print("-" * 93)
        
        # Low estimator
        Price_low = np.mean(results[:, 1])
        std_low = np.std(results[:, 1])
        error_low = 100 * 2 * 1.96 * std_low / (np.sqrt(self.params.n) * Price_low)
        ci_low = [Price_low - 1.96 * std_low / np.sqrt(self.params.n),
                 Price_low + 1.96 * std_low / np.sqrt(self.params.n)]
        
        print("Low Estimator price Option =", Price_low, "Error =", error_low, "%")
        print("Low Confidence interval at 95% =", ci_low)
        print("-" * 93)
        
        # Final estimation
        Price_estimation = (ci_low[0] + ci_high[1]) / 2
        error_estimation = -100 * (ci_low[0] - ci_high[1]) / Price_estimation
        
        print("Estimation price Option =", Price_estimation, "Error =", error_estimation, "%")
        print("Confidence interval at 95% = [", ci_low[0], ",", ci_high[1], "]")
        print("-" * 93)

# Example usage
params = PricingParams()
pricer = OptimizedBermudeanPricerCV(params)
results, computation_time = pricer.price_option()
pricer.print_results(results, computation_time)


Numerical results
Time Execution = 1.839600999839604 s
---------------------------------------------------------------------------------------------
High Estimator price Option = 8.20569325913136 ErrorMargin = 12.125472166306801 %
High Confidence interval at 95% = [7.7082037330371165, 8.703182785225604]
---------------------------------------------------------------------------------------------
Low Estimator price Option = 8.11689697381029 Error = 12.264557484869602 %
Low Confidence interval at 95% = [7.619146226139987, 8.614647721480592]
---------------------------------------------------------------------------------------------
Estimation price Option = 8.161164505682795 Error = 13.28286616855693 %
Confidence interval at 95% = [ 7.619146226139987 , 8.703182785225604 ]
---------------------------------------------------------------------------------------------


### ALL COMBINED: Pruning, Control variates, Antithetic variables

In [12]:
import numpy as np
import scipy.stats as sps
import time
from dataclasses import dataclass
from typing import List, Tuple, Optional

@dataclass
class PricingParams:
    """Parameters for Bermudean put option pricing"""
    S_0: float = 100.0    # Initial stock price
    K: float = 100.0      # Strike price
    r: float = 0.05       # Risk-free rate
    sigma: float = 0.2    # Volatility
    T: float = 3.0        # Time to maturity
    m: int = 3            # Number of exercise dates
    b: int = 50           # Number of branches for pruning
    n: int = 500          # Number of simulations

class OptimizedBermudeanPricer:
    """
    Enhanced Bermudean put option pricer using pruning and antithetic variables
    for variance reduction and improved computational efficiency.
    """
    
    def __init__(self, params: PricingParams):
        self.params = params
        self.dt = params.T / params.m
        
        # Precompute constants for efficiency
        self.drift = (params.r - 0.5 * params.sigma**2) * self.dt
        self.vol = params.sigma * np.sqrt(self.dt)
        self.discount = np.exp(-params.r * self.dt)
        
        # Precompute normal distribution for performance
        self._norm = sps.norm()
        
    def _black_scholes_put(self, S: float, time_to_maturity: float) -> float:
        """
        Optimized Black-Scholes formula for European put pricing
        
        Args:
            S: Current stock price
            time_to_maturity: Time remaining until maturity
            
        Returns:
            float: European put option price
        """
        if time_to_maturity <= 0:
            return max(self.params.K - S, 0)
            
        sqrt_t = np.sqrt(time_to_maturity)
        d1 = (np.log(S/self.params.K) + 
              (self.params.r + 0.5 * self.params.sigma**2) * time_to_maturity) / \
             (self.params.sigma * sqrt_t)
        d2 = d1 - self.params.sigma * sqrt_t
        
        return (self.params.K * np.exp(-self.params.r * time_to_maturity) * 
                self._norm.cdf(-d2) - S * self._norm.cdf(-d1))
    
    def _generate_paths(self, S: float, size: int) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate antithetic price paths for variance reduction
        
        Args:
            S: Current stock price
            size: Number of paths to generate
            
        Returns:
            Tuple containing normal and antithetic paths
        """
        Z = np.random.standard_normal(size)
        paths = S * np.exp(self.drift + self.vol * Z)
        paths_antithetic = S * np.exp(self.drift - self.vol * Z)
        return paths, paths_antithetic
    
    def _low_estimator(self, V_successor: np.ndarray, S1: float, S2: Optional[float] = None) -> float:
        """
        Calculate the low estimator value.
    
        Args:
            V_successor: Successor node values, a 2D array of shape (b, 2).
            S1: Initial price of the first underlying asset.
            S2: Initial price of the second underlying asset (optional).
    
        Returns:
            The low estimator value for the given node.
        """
        V = np.zeros(self.params.b)
        for j in range(self.params.b):
            # Define the ensemble excluding the current index
            ensemble = [i for i in range(self.params.b) if i != j]
    
            # Calculate V1 and V2 based on successor values
            V1 = np.exp(-self.params.r * self.dt) * np.mean(V_successor[ensemble, 1])
            V2 = np.exp(-self.params.r * self.dt) * V_successor[j, 1]
    
            # Determine immediate exercise value
            if S2 is None:
                h = max(self.params.K - S1, 0)
            else:
                h = np.mean([max(self.params.K - S1, 0), max(self.params.K - S2, 0)])
    
            # Apply the low estimator logic
            V[j] = h if V1 <= h else V2
    
        # Return the average value as the estimator
        return np.mean(V)
    
    def estimate_node(self, S1: float, S2: float, k: int) -> List[float]:
        """
        Estimate option value at a given node using both price paths
        
        Args:
            S1: First price path
            S2: Antithetic price path
            k: Current time step
            
        Returns:
            List containing [high estimate, low estimate]
        """
        time_to_maturity = (self.params.m - k) * self.dt
        immediate_exercise1 = max(self.params.K - S1, 0)
        immediate_exercise2 = max(self.params.K - S2, 0)
        immediate_exercise = 0.5 * (immediate_exercise1 + immediate_exercise2)
        
        # Terminal condition
        if k == self.params.m - 1:
            european_value = 0.5 * (self._black_scholes_put(S1, time_to_maturity) + 
                                  self._black_scholes_put(S2, time_to_maturity))
            value = max(immediate_exercise, european_value)
            return [value, value]
            
        # Check pruning condition using European value
        european_value = 0.5 * (self._black_scholes_put(S1, time_to_maturity) + 
                               self._black_scholes_put(S2, time_to_maturity))
        
        if immediate_exercise < european_value:
            # Continue with single successor pair
            next_S1, next_S2 = self._generate_paths(S1, 1)[0][0], self._generate_paths(S2, 1)[0][0]
            successor_values = self.estimate_node(next_S1, next_S2, k + 1)
            return [self.discount * v for v in successor_values]
        
        # Generate full set of successors
        V_successor = np.zeros((self.params.b, 2))
        for i in range(self.params.b):
            paths1, paths2 = self._generate_paths(S1, 1)[0][0], self._generate_paths(S2, 1)[0][0]
            V_successor[i, :] = self.estimate_node(paths1, paths2, k + 1)
        
        # Calculate estimators with variance reduction
        continuation_value = self.discount * np.mean(V_successor[:, 0])
        V_high = max(immediate_exercise, continuation_value)
        V_low = self._low_estimator(V_successor, S1, S2)
        
        return [V_high, V_low]

    def price_option(self) -> Tuple[np.ndarray, float]:
        """
        Price the Bermudean put option using Monte Carlo simulation
        
        Returns:
            Tuple containing (Results array, Computation time)
        """
        start_time = time.perf_counter()
        
        results = np.zeros((self.params.n, 2))
        for i in range(self.params.n):
            results[i, :] = self.estimate_node(self.params.S_0, self.params.S_0, 0)
            
        computation_time = time.perf_counter() - start_time
        return results, computation_time
    
    def print_results(self, results: np.ndarray, computation_time: float) -> None:
        """
        Print detailed pricing results with confidence intervals
        
        Args:
            results: Array of simulation results
            computation_time: Total computation time
        """
        print("\nNumerical results")
        print(f"Time Execution = {computation_time} s")
        print("-" * 93)
        
        # High estimator calculations
        Price_high = np.mean(results[:, 0])
        std_high = np.std(results[:, 0])
        error_high = 100 * 2 * 1.96 * std_high / (np.sqrt(self.params.n) * Price_high)
        ci_high = [Price_high - 1.96 * std_high / np.sqrt(self.params.n),
                  Price_high + 1.96 * std_high / np.sqrt(self.params.n)]
        
        print(f"High Estimator price Option = {Price_high} ErrorMargin = {error_high}%")
        print(f"High Confidence interval at 95% = {ci_high}")
        print("-" * 93)
        
        # Low estimator calculations
        Price_low = np.mean(results[:, 1])
        std_low = np.std(results[:, 1])
        error_low = 100 * 2 * 1.96 * std_low / (np.sqrt(self.params.n) * Price_low)
        ci_low = [Price_low - 1.96 * std_low / np.sqrt(self.params.n),
                 Price_low + 1.96 * std_low / np.sqrt(self.params.n)]
        
        print(f"Low Estimator price Option = {Price_low} Error = {error_low}%")
        print(f"Low Confidence interval at 95% = {ci_low}")
        print("-" * 93)
        
        # Final price estimation
        Price_estimation = (ci_low[0] + ci_high[1]) / 2
        error_estimation = -100 * (ci_low[0] - ci_high[1]) / Price_estimation
        ci_final = [ci_low[0], ci_high[1]]
        
        print(f"Estimation price Option = {Price_estimation} Error = {error_estimation}%")
        print(f"Confidence interval at 95% = {ci_final}")
        print("-" * 93)

params = PricingParams()
pricer = OptimizedBermudeanPricer(params)
results, computation_time = pricer.price_option()
pricer.print_results(results, computation_time)


Numerical results
Time Execution = 4.376858700066805 s
---------------------------------------------------------------------------------------------
High Estimator price Option = 7.916266251398294 ErrorMargin = 14.498936792698107%
High Confidence interval at 95% = [7.342379031332329, 8.490153471464259]
---------------------------------------------------------------------------------------------
Low Estimator price Option = 7.840620585583273 Error = 14.539286517065387%
Low Confidence interval at 95% = [7.270635439756292, 8.410605731410254]
---------------------------------------------------------------------------------------------
Estimation price Option = 7.880394455610276 Error = 15.47534249176775%
Confidence interval at 95% = [7.270635439756292, 8.490153471464259]
---------------------------------------------------------------------------------------------
